# Deep PILCO — passer à l'échelle avec un réseau bayésien

> **Deep PILCO** (Gal, McAllister & Rasmussen, 2016) = *PILCO original, mais le Gaussian Process
> est remplacé par un **réseau de neurones bayésien** (BNN), et la propagation analytique par une
> propagation **par particules**.*

C'est le **second** notebook du diptyque. Le premier [10_pilco_walkthrough](./10_pilco_walkthrough.ipynb) a construit, brique
par brique, le PILCO classique : GP de dynamique, moment matching analytique, coût saturant,
politique RBF, boucle Observer → Imaginer → Ajuster. **On suppose ces idées acquises.**

Ici, on ne change que **deux briques** de cette boucle — le *modèle* et la *propagation* — tout en
gardant l'esprit. Le but : comprendre *pourquoi* on a besoin de Deep PILCO, et *comment* un réseau
bayésien + des particules remplacent le GP + le moment matching analytique.

## 1. Pourquoi remplacer le GP ? Le mur du $O(n^3)$

Le GP est magnifique mais il a un coût rédhibitoire. Prédire ou ajuster un GP demande de
factoriser la matrice $K + \sigma_n^2 I$ de taille $n \times n$ (où $n$ = nombre de transitions) :

- temps : $O(n^3)$,
- mémoire : $O(n^2)$.

Conséquence concrète :

- quelques **centaines** de points : très bien (le régime de PILCO sur le pendule) ;
- quelques **milliers** : déjà pénible ;
- **haute dimension** + dizaines de milliers de points : pratiquement impossible.

**Analogie.** PILCO avec GP **mémorise** beaucoup de cas particuliers (il garde toutes les
données) ; un réseau de neurones **apprend une règle générale** qui se transfère mieux et dont le
coût d'inférence ne grossit pas avec les données. Mesurons ce mur du $O(n^3)$.

In [ ]:
import time
from pathlib import Path

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Video, display
from tqdm.auto import tqdm

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
# Coût d'une factorisation de Cholesky n×n (le goulot du GP) en fonction de n
ns = [50, 100, 200, 400, 800, 1600]
times = []
for n in ns:
    A = np.random.randn(n, n); K = A @ A.T + n * np.eye(n)
    t0 = time.perf_counter()
    for _ in range(3): np.linalg.cholesky(K)
    times.append((time.perf_counter() - t0) / 3)
plt.figure(figsize=(7.5, 3.6))
plt.loglog(ns, times, "o-", label="temps mesuré")
plt.loglog(ns, [times[0] * (n / ns[0]) ** 3 for n in ns], "--", color="gray", label="pente $O(n^3)$")
plt.xlabel("nombre de points n"); plt.ylabel("temps de factorisation (s)")
plt.title("Le GP se heurte au mur du $O(n^3)$"); plt.legend(); plt.show()


**Lecture.** Le temps suit la pente $O(n^3)$ (pointillés) : multiplier les données par 10
multiplie le coût par 1000. Un GP est donc cantonné à quelques centaines de points. Pour des tâches
plus grosses ou plus longues, il faut un modèle dont l'inférence **ne dépend pas** de la taille du
jeu de données : un **réseau de neurones**. Mais un réseau ordinaire perd ce qui faisait toute la
force de PILCO — **l'incertitude calibrée**. Il faut donc un réseau qui sache, lui aussi, dire
« je ne sais pas » : un **réseau bayésien**.

## I.1 — D'un réseau déterministe à une distribution sur les poids

Un **réseau classique** apprend des poids **fixes** $W$ par descente de gradient :

$$y = f(x, W).$$

Pour une entrée donnée il renvoie **un seul** nombre, sans aucune notion de confiance. Le piège : un
réseau ReLU **extrapole de façon arbitraire** hors de ses données — interrogé loin de ce qu'il a vu,
il ne se tait pas, il répond avec le même aplomb qu'au cœur de ses données. En planification
model-based, cette **surconfiance silencieuse** est fatale : le planificateur fait confiance à une
prédiction inventée, propage l'erreur sur tout l'horizon, et optimise une politique pour un monde qui
n'existe pas.

Un **réseau bayésien (BNN)** remplace le point unique $W$ par une **distribution** sur les poids :

$$W \sim p(W \mid \mathcal{D}), \qquad y = f(x, W).$$

On ne cherche plus *le* réglage parfait, mais **toute la famille des réglages plausibles** compatibles
avec les données. Prédire devient alors un acte statistique en deux temps :

$$\underbrace{\mathbb{E}_{W}[\,f(x,W)\,]}_{\text{prédiction}}, \qquad
\underbrace{\mathrm{Var}_{W}[\,f(x,W)\,]}_{\text{incertitude épistémique}}.$$

- là où **tous** les réseaux plausibles s'accordent → variance faible → on est confiant ;
- là où ils **divergent** → variance forte → le modèle avoue son ignorance.

Cette variance est dite **épistémique** : elle vient du manque de données et **décroît** quand on en
collecte davantage — exactement le signal dont PILCO a besoin pour savoir *où explorer*.

**Analogie.** Réseau classique = **un seul expert** qui tranche toujours, même hors de son domaine.
BNN = **un comité d'experts** tous compatibles avec les données : sur le terrain connu ils répondent
à l'unisson ; sur une question nouvelle ils se contredisent, et ce désaccord *est* la mesure
d'incertitude.

## I.2 — Le posterior sur les poids est insoluble

Le théorème de Bayes donne, en principe, la distribution des poids sachant les données :

$$p(W \mid \mathcal{D}) = \frac{p(\mathcal{D} \mid W)\, p(W)}{p(\mathcal{D})},
\qquad p(\mathcal{D}) = \int p(\mathcal{D}\mid W)\,p(W)\,dW.$$

| Terme | Sens | Calculable ? |
|-------|------|--------------|
| $p(\mathcal{D}\mid W)$ | **vraisemblance** : à quel point ces poids expliquent les données | ✅ une passe avant |
| $p(W)$ | **a priori** sur les poids (gaussien ⇔ régularisation L2) | ✅ trivial |
| $p(\mathcal{D})$ | **évidence** : intégrale sur **tous** les réglages de poids possibles | ❌ insoluble |

Le verrou est le dénominateur $p(\mathcal{D})$ : une intégrale en dimension *millions* (un paramètre
par poids), sur un intégrande non gaussien et multimodal, sans forme close ni quadrature praticable à
cette échelle. Sans lui, impossible de **normaliser** — donc d'obtenir — le posterior exactement.

Deux familles d'approximations le contournent :

- **MCMC** : on *échantillonne* le posterior sans jamais calculer $p(\mathcal{D})$ (il se simplifie
  dans les ratios d'acceptation). Asymptotiquement exact, mais **lent** et difficile à diagnostiquer
  en haute dimension.
- **Inférence variationnelle (VI)** : on remplace le vrai posterior par une famille simple
  $q_\theta(W)$ et on **optimise** $\theta$ pour la rapprocher de $p(W\mid\mathcal{D})$, en maximisant
  l'**ELBO** :
  $$\mathcal{L}(\theta) = \underbrace{\mathbb{E}_{q_\theta}\!\big[\log p(\mathcal{D}\mid W)\big]}_{\text{coller aux données}}
  - \underbrace{\mathrm{KL}\!\big(q_\theta(W)\,\|\,p(W)\big)}_{\text{rester près de l'a priori}}.$$
  Rapide, mais l'incertitude reste **approchée** (bornée par l'expressivité de $q_\theta$).

Deep PILCO utilise une astuce d'inférence variationnelle remarquablement simple — le **dropout** —
qu'on déroule en partie II.

## I.3 — GP vs BNN : deux façons d'être incertain

GP et BNN incarnent la **même** idée — une distribution *sur les fonctions* — par deux chemins opposés :

- le **GP** met une distribution directement sur les **fonctions** (objet de dimension infinie), via
  un noyau : incertitude **exacte**, mais coût $O(n^3)$ qui étouffe quand les données abondent;
- le **BNN** met une distribution sur les **poids** (grande dimension, mais **finie**), ce qui *induit*
  une distribution sur les fonctions : incertitude **approchée**, mais coût linéaire, GPU-friendly,
  intégré nativement au deep learning.

| | **GP** | **BNN (MC-dropout)** |
|---|---|---|
| Distribution sur… | les fonctions | les poids → les fonctions |
| Incertitude | exacte (forme close) | approchée (échantillonnage) |
| Passage à l'échelle | $O(n^3)$, souffre si $n$ grand | linéaire en $n$, GPU |
| Hyperparamètres | noyau interprétable | architecture + taux de dropout |
| Dans la lignée | PILCO (2011) | Deep PILCO (2016) |

Le GP est le choix élégant du PILCO original (peu de données, incertitude exacte propagée
analytiquement) ; le BNN est le pari de Deep PILCO — sacrifier l'exactitude de l'incertitude pour
gagner l'échelle et la flexibilité. On les compare côte à côte en partie III. Mais avant toute
machinerie, **matérialisons l'idée du « comité d'experts »** de §I.1 avec un simple ensemble de
réseaux.

In [ ]:
# Intuition de §I.1 : un "comité d'experts" = un ensemble de réseaux déterministes
# entraînés INDÉPENDAMMENT (init différente). Là où ils s'accordent -> confiance ;
# là où ils divergent -> incertitude. C'est l'idée "distribution sur les poids ->
# distribution sur les fonctions", rendue visible sans aucune machinerie bayésienne.
torch.manual_seed(0); np.random.seed(0)

def true_fn(x):
    return np.sin(1.5 * x)

# Deux paquets de données avec un TROU au milieu : le terrain inconnu où il faut douter.
x_left  = np.random.uniform(-3.0, -1.2, 25)
x_right = np.random.uniform( 1.2,  3.0, 25)
x_train = np.concatenate([x_left, x_right])
y_train = true_fn(x_train) + 0.05 * np.random.randn(x_train.size)
Xtr_e = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
Ytr_e = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

def make_mlp():
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))

M = 15                                            # 15 experts plausibles
experts = []
for m in range(M):
    torch.manual_seed(m)                          # une initialisation différente par expert
    net_m = make_mlp(); opt_m = torch.optim.Adam(net_m.parameters(), lr=1e-2)
    for _ in range(800):
        opt_m.zero_grad(); F.mse_loss(net_m(Xtr_e), Ytr_e).backward(); opt_m.step()
    experts.append(net_m)

grid_e = torch.linspace(-5, 5, 300).unsqueeze(1)
with torch.no_grad():
    preds = torch.stack([net_m(grid_e)[:, 0] for net_m in experts])   # [M, 300]
mu_e, std_e = preds.mean(0), preds.std(0)

g = grid_e[:, 0].numpy(); std_np = std_e.numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax in axes:
    ax.scatter(x_train, y_train, c="red", s=14, zorder=5, label="données")
    ax.plot(g, true_fn(g), "g--", lw=1.3, label="fonction vraie")
    ax.axvspan(-1.2, 1.2, color="orange", alpha=0.08); ax.set_ylim(-2.5, 2.5); ax.set_xlabel("x")
# Gauche : UN seul réseau -> une courbe nette partout, aucune notion de doute.
axes[0].plot(g, preds[0].numpy(), "b", lw=2, label="un seul réseau")
axes[0].set_title("Un expert unique (confiant même dans le trou)")
axes[0].legend(fontsize=8, loc="upper left")
# Droite : le COMITÉ -> moyenne + désaccord (±2σ) qui s'évase loin des données.
axes[1].plot(g, mu_e.numpy(), "b", lw=2, label="moyenne du comité")
axes[1].fill_between(g, (mu_e - 2 * std_e).numpy(), (mu_e + 2 * std_e).numpy(),
                     color="b", alpha=0.18, label="±2σ (désaccord)")
axes[1].set_title("Un comité d'experts (doute dans le trou et au-delà)")
axes[1].legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

print(f"désaccord σ — sur les données (|x| in [1.2, 3]) : {std_np[(g >= 1.2) & (g <= 3)].mean():.3f}")
print(f"désaccord σ — dans le trou    (|x| < 1.2)       : {std_np[np.abs(g) < 1.2].mean():.3f}")


**Lecture.** À **gauche**, un réseau unique trace une courbe nette *partout* — y compris dans le
trou orange où il n'a aucune donnée : il ne distingue pas ce qu'il *sait* de ce qu'il *invente*,
c'est la surconfiance dénoncée en §I.1. À **droite**, le **comité** de réseaux indépendants donne la
même moyenne près des données mais une bande ±2σ qui **s'évase** dans le trou et en extrapolation :
le désaccord des experts *est* l'incertitude épistémique. Les deux `σ` imprimés le confirment —
faible sur les données, nettement plus grand dans le trou.

⚠️ Ici le « comité » est un **ensemble** : $M$ réseaux entraînés séparément, donc $M$ entraînements
complets — coûteux. La partie II obtient le **même** effet à coût quasi nul, avec **un seul** réseau
et du **dropout** maintenu actif au moment de la prédiction : c'est le **MC-dropout**, l'inférence
variationnelle annoncée en §I.2.

---
# Partie II — MC-dropout : un BNN à très peu de frais

## II.1 — Le dropout, vu comme de l'inférence bayésienne

Le **dropout** est connu comme régularisation : à l'entraînement, on éteint aléatoirement une
fraction $p$ des neurones à chaque passe. [Gal & Ghahramani (2016)](https://arxiv.org/pdf/1506.02142) ont montré qu'il s'agit en fait
d'une **inférence variationnelle approchée** : entraîner avec dropout revient à approcher le
posterior sur les poids.

La recette, dite **MC-dropout**, est alors d'une simplicité déconcertante : **garder le dropout
actif au moment de la prédiction** et faire plusieurs passes. Chaque passe utilise un masque de
Bernoulli différent

$$\text{dropout}(h) = h \odot m, \qquad m_i \sim \text{Bernoulli}(1-p),$$

ce qui correspond à **échantillonner un réseau différent** du posterior approché. Chaque masque
$M^{(k)}$ définit un « expert » du comité :

$$y^{(k)} = f(x, W \odot M^{(k)}).$$

En agrégeant $K$ passes, on estime moyenne **et** incertitude :

$$\mu_y \approx \frac{1}{K}\sum_{k=1}^{K} y^{(k)}, \qquad
\Sigma_y \approx \frac{1}{K}\sum_{k=1}^{K} (y^{(k)} - \mu_y)(y^{(k)} - \mu_y)^\top.$$

Le seul hyperparamètre nouveau est le taux $p$ : trop grand, le réseau sous-apprend ; trop petit,
l'incertitude est sous-estimée.

In [ ]:
class BayesianDynamicsNet(nn.Module):
    # MLP (x,u) -> Δx, avec dropout par masques de Bernoulli EXPLICITES.
    # Garder un masque fixe permet à une particule de voir un réseau cohérent sur tout l'horizon.
    def __init__(self, input_dim, output_dim, hidden=64, n_layers=2, p=0.1):
        super().__init__()
        dims = [input_dim] + [hidden] * n_layers
        self.linears = nn.ModuleList([nn.Linear(dims[i], dims[i + 1]) for i in range(n_layers)])
        self.out = nn.Linear(hidden, output_dim)
        self.hidden, self.n_layers, self.p = hidden, n_layers, p
        self.scale = 1.0 / (1.0 - p)  # dropout inversé : garde l'espérance de l'activation

    def sample_masks(self, K):
        # Un masque [K, hidden] par couche cachée.
        # K peut être un batch de données, ou plus tard un nombre de particules.
        return [
            torch.bernoulli(torch.full((K, self.hidden), 1.0 - self.p))
            for _ in range(self.n_layers)
        ]

    def forward(self, xu, masks=None):
        if masks is None:
            masks = self.sample_masks(xu.shape[0])  # masques frais : un "réseau" échantillonné

        h = xu
        for i, lin in enumerate(self.linears):
            h = F.silu(lin(h))
            h = h * masks[i] * self.scale
        return self.out(h)


# Mini-test visuel : même masque -> même sortie ; masques différents -> sorties différentes.
torch.manual_seed(0)
net = BayesianDynamicsNet(2, 1, hidden=32, p=0.2)

# On prend un petit batch de 4 entrées fixes.
x = torch.randn(4, 2)

# Cas A : même masque réutilisé deux fois.
fixed_masks = net.sample_masks(K=x.shape[0])
y_fixed_1 = net(x, fixed_masks).detach().squeeze(-1)
y_fixed_2 = net(x, fixed_masks).detach().squeeze(-1)

# Cas B : nouveaux masques à chaque passe.
y_random_1 = net(x, net.sample_masks(K=x.shape[0])).detach().squeeze(-1)
y_random_2 = net(x, net.sample_masks(K=x.shape[0])).detach().squeeze(-1)

assert torch.allclose(y_fixed_1, y_fixed_2), "même masque doit donner exactement la même sortie"
assert not torch.allclose(y_random_1, y_random_2), "masques différents doivent changer la sortie"

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

# 1) Visualiser les entrées du batch.
axes[0].scatter(x[:, 0], x[:, 1], s=80, color="black")
for i, (x0, x1) in enumerate(x.numpy()):
    axes[0].annotate(f"#{i}", (x0, x1), xytext=(6, 6), textcoords="offset points")
axes[0].set_title("Batch fixe d'entrées $(x,u)$")
axes[0].set_xlabel("dimension 1")
axes[0].set_ylabel("dimension 2")

# 2) Même masque : les deux sorties se superposent exactement.
idx = np.arange(x.shape[0])
axes[1].plot(idx, y_fixed_1.numpy(), "o-", lw=2, label="passe 1")
axes[1].plot(idx, y_fixed_2.numpy(), "x--", lw=2, label="passe 2")
axes[1].set_title("Même masque dropout\nsortie déterministe")
axes[1].set_xlabel("index de l'entrée")
axes[1].set_ylabel("sortie prédite")
axes[1].legend()

# 3) Masques différents : les sorties changent.
axes[2].plot(idx, y_random_1.numpy(), "o-", lw=2, label="masque A")
axes[2].plot(idx, y_random_2.numpy(), "x--", lw=2, label="masque B")
axes[2].set_title("Masques différents\nréseaux échantillonnés différents")
axes[2].set_xlabel("index de l'entrée")
axes[2].set_ylabel("sortie prédite")
axes[2].legend()

plt.tight_layout()
plt.show()

print("✓ BNN OK : déterministe à masque fixé, stochastique à masques rééchantillonnés")

**Lecture.** Le panneau de gauche fixe le batch d'entrées : on interroge toujours le réseau aux
mêmes points. Au centre, on réutilise exactement les mêmes masques de dropout pour deux passes :
les sorties se superposent. Cela montre qu'un masque fixé définit un réseau déterministe, donc une
dynamique cohérente.

À droite, on change les masques entre les deux passes. Les sorties changent alors, même si les
entrées sont identiques. C'est l'idée MC-dropout : chaque masque échantillonne un membre différent
du comité de modèles. Deep PILCO utilisera cette propriété pour donner à chaque particule sa propre
dynamique plausible pendant tout le rollout.

In [ ]:
# Entraîner le BNN sur une fonction jouet, puis 100 passes MC-dropout = un comité d'experts
torch.manual_seed(0)
Xtr = torch.linspace(-3, 3, 30).unsqueeze(1)
ytr = torch.sin(Xtr) + 0.05 * torch.randn_like(Xtr)
bnn = BayesianDynamicsNet(1, 1, hidden=64, p=0.1)
opt = torch.optim.Adam(bnn.parameters(), lr=1e-2)
for _ in range(1500):
    opt.zero_grad(); loss = F.mse_loss(bnn(Xtr), ytr); loss.backward(); opt.step()

grid = torch.linspace(-5, 5, 200).unsqueeze(1)
with torch.no_grad():
    preds = torch.stack([bnn(grid)[:, 0] for _ in range(100)])   # 100 réseaux échantillonnés
mu, std = preds.mean(0), preds.std(0)
plt.figure(figsize=(9, 4))
for k in range(40):
    plt.plot(grid[:, 0], preds[k], color="steelblue", alpha=0.08)
plt.plot(grid[:, 0], mu, "b", lw=2, label="moyenne du comité")
plt.fill_between(grid[:, 0], mu - 2 * std, mu + 2 * std, alpha=0.2, color="orange", label="±2σ (désaccord)")
plt.scatter(Xtr[:, 0], ytr[:, 0], c="red", s=15, zorder=5, label="données")
plt.title("MC-dropout : 100 réseaux échantillonnés (spaghetti)"); plt.xlabel("x"); plt.legend(); plt.show()

**Lecture.** Chaque trait bleu pâle est **un** réseau tiré par dropout — un membre du comité.
Là où il y a des données (centre), les experts **s'accordent** : faisceau serré, faible
incertitude. Loin des données (bords), ils **divergent** : le faisceau s'évase, l'incertitude
(orange) grossit. On retrouve **qualitativement** le comportement du GP du notebook 10 — certain
près des données, incertain ailleurs — mais obtenu avec un simple réseau + dropout, qui lui passe
à l'échelle. Le taux $p$ règle l'ampleur de ce désaccord.

In [ ]:
# Effet du taux de dropout p sur l'incertitude
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, p in zip(axes, [0.02, 0.1, 0.3]):
    torch.manual_seed(0)
    b = BayesianDynamicsNet(1, 1, hidden=64, p=p); o = torch.optim.Adam(b.parameters(), lr=1e-2)
    for _ in range(1200):
        o.zero_grad(); F.mse_loss(b(Xtr), ytr).backward(); o.step()
    with torch.no_grad():
        pr = torch.stack([b(grid)[:, 0] for _ in range(80)])
    m, s = pr.mean(0), pr.std(0)
    ax.plot(grid[:, 0], m, "b"); ax.fill_between(grid[:, 0], m - 2 * s, m + 2 * s, alpha=0.2, color="orange")
    ax.scatter(Xtr[:, 0], ytr[:, 0], c="red", s=10); ax.set_title(f"p = {p}"); ax.set_xlabel("x")
plt.suptitle("Le taux de dropout p contrôle l'ampleur de l'incertitude"); plt.tight_layout(); plt.show()

**Lecture.** $p$ trop petit (gauche) : le comité est presque unanime partout — l'incertitude
est **sous-estimée**, le réseau reste trop confiant loin des données. $p$ trop grand (droite) : le
désaccord est large même sur les données, le réseau **sous-apprend**. Il y a un réglage
intermédiaire (centre) où l'incertitude est bien calibrée. C'est l'analogue, pour le BNN, du choix
des hyperparamètres du GP — sauf qu'ici il se règle souvent par validation croisée.

---
# Partie III — Propager l'incertitude : les particules

## III.1 — Pourquoi on ne peut plus faire de moment matching analytique

Dans PILCO classique, on propage une croyance gaussienne sur l'état :

$$
x_t \sim \mathcal N(\mu_t, \Sigma_t).
$$

Le modèle de dynamique est un GP avec un kernel SE/RBF. Cette structure est très particulière :
elle permet de calculer analytiquement les moments de la sortie. Autrement dit, on peut demander :

> si l'entrée du GP est gaussienne, quelle est la moyenne et la covariance de la sortie ?

et obtenir une réponse en forme fermée. C'est le **moment matching analytique** du notebook 10.

Avec Deep PILCO, on remplace le GP par un réseau de neurones bayésien :

$$
\Delta x_t = f_\phi(x_t, u_t).
$$

Ce réseau contient des couches linéaires, des activations non-linéaires, du dropout, et une politique
elle-même non-linéaire. On voudrait encore calculer :

$$
\mathbb E[f_\phi(x_t, \pi(x_t))]
\qquad\text{et}\qquad
\operatorname{Cov}(f_\phi(x_t, \pi(x_t))),
$$

mais cette fois l'intégrale n'a plus de forme fermée exploitable. La transformation est trop
complexe : une gaussienne qui passe dans un MLP ne ressort pas comme une gaussienne simple dont on
peut écrire les moments à la main.

**Analogie.** Dans PILCO classique, on faisait passer une boule de pâte parfaitement ronde dans une
machine dont on connaissait les équations : on pouvait prédire exactement la forme moyenne à la
sortie. Dans Deep PILCO, la machine est un assemblage de couches non-linéaires et de masques
dropout : la pâte peut être étirée, pliée, tordue. Plutôt que de calculer la forme finale avec une
formule impossible, on suit beaucoup de petits points de pâte et on regarde où ils arrivent.

C'est l'idée des **particules**.

Au lieu de représenter la croyance uniquement par ses deux paramètres gaussiens,

$$
(\mu_t, \Sigma_t),
$$

on la représente par un nuage de $K$ échantillons :

$$
\{x_t^{(1)}, x_t^{(2)}, \dots, x_t^{(K)}\}.
$$

Chaque particule est une hypothèse possible sur l'état réel du système. Si la croyance initiale est
concentrée, les particules sont proches. Si l'incertitude est grande, elles sont dispersées.

Ensuite, on pousse chaque particule dans la boucle modèle + politique :

1. partir de $K$ particules $\{x_t^{(k)}\}_{k=1}^K$ ;
2. calculer l'action de chaque particule :

$$
u_t^{(k)} = \pi(x_t^{(k)});
$$

3. prédire le delta d'état avec le BNN :

$$
\Delta x_t^{(k)} = f_\phi(x_t^{(k)}, u_t^{(k)});
$$

4. avancer la particule :

$$
x_{t+1}^{(k)} = x_t^{(k)} + \Delta x_t^{(k)};
$$

5. estimer le coût par moyenne Monte-Carlo :

$$
\mathbb E[c(x_{t+1})]
\approx
\frac{1}{K}\sum_{k=1}^{K} c(x_{t+1}^{(k)}).
$$

C'est l'analogue numérique du moment matching. PILCO calculait directement la moyenne et la
covariance avec des intégrales fermées. Deep PILCO remplace ces intégrales par un nuage de
particules.

On peut résumer ainsi :

| PILCO classique | Deep PILCO |
|---|---|
| croyance représentée par $(\mu,\Sigma)$ | croyance représentée par des particules |
| propagation analytique dans le GP | propagation numérique dans le BNN |
| moment matching en forme fermée | moment matching Monte-Carlo |
| très élégant mais contraint | plus flexible mais bruité |

L'avantage est énorme : on peut maintenant propager l'incertitude à travers presque n'importe quel
réseau différentiable. La contrepartie est que le résultat devient approximatif : avec peu de
particules, l'estimation est bruitée ; avec beaucoup de particules, elle coûte plus cher.

Deux subtilités font toute la qualité de Deep PILCO :

1. **comment on échantillonne le réseau pour chaque particule** : c'est le rôle des masques de
   dropout fixes de la partie III.2 ;
2. **comment on garde le nuage tractable après chaque pas** : c'est le resample par moment matching
   numérique de la partie III.3.

Sans ces deux détails, la méthode des particules peut donner une illusion d'incertitude tout en
sous-estimant fortement le risque réel du modèle.


## III.2 — Le point clé : un masque de dropout **fixe** par particule

Voici l'idée la plus subtile de Deep PILCO : pendant un rollout imaginaire, une particule ne doit
pas changer de modèle à chaque pas.

Avec MC-dropout, chaque masque de dropout correspond à un **réseau échantillonné** dans le posterior
approché. On peut donc lire un masque comme une hypothèse possible sur la dynamique réelle :

$$
M^{(k)} \quad \Longleftrightarrow \quad f_{\phi, M^{(k)}}.
$$

Une particule représente une trajectoire possible du système. Si cette particule utilise le masque
$M^{(k)}$, alors elle suit la dynamique :

$$
x_{t+1}^{(k)}
=
x_t^{(k)}
+
f_{\phi, M^{(k)}}(x_t^{(k)}, u_t^{(k)}).
$$

Le point crucial est que ce même masque doit rester fixé pendant tout l'horizon :

$$
M^{(k)}_0 = M^{(k)}_1 = \dots = M^{(k)}_{H-1}.
$$

Pourquoi ? Parce que la vraie dynamique du monde est **une seule fonction**, même si on ne sait pas
exactement laquelle. Si le modèle réel est un peu plus rapide, plus lent, plus optimiste ou plus
pessimiste que notre moyenne, cette erreur ne disparaît pas magiquement au pas suivant. Elle reste
cohérente et se propage dans le temps.

Avec un masque fixe, chaque particule dit donc :

> supposons que le monde ressemble à cette dynamique plausible ; que devient la trajectoire ?

Une autre particule reçoit un autre masque, donc une autre dynamique plausible. Le nuage complet
représente alors plusieurs mondes possibles.

---

### Ce qui se passe si on rééchantillonne à chaque pas

Si on tire un nouveau masque à chaque instant, une même particule change de modèle au milieu de sa
trajectoire :

$$
x_{t+1}^{(k)}
=
x_t^{(k)}
+
f_{\phi, M_t^{(k)}}(x_t^{(k)}, u_t^{(k)}),
\qquad
M_t^{(k)} \neq M_{t+1}^{(k)}.
$$

Cela revient à dire que le monde change de lois physiques à chaque pas de temps. Un pas est simulé
par un modèle optimiste, le suivant par un modèle pessimiste, puis un autre par un modèle moyen. Les
erreurs se mélangent et se compensent artificiellement.

Le résultat est dangereux : l'incertitude de trajectoire semble plus petite qu'elle ne devrait
l'être. Le planner croit que le futur est stable, alors qu'il a simplement moyenné des erreurs qui
auraient dû rester corrélées.

---

### Analogie

Imagine qu'on demande à plusieurs ingénieurs de prédire la trajectoire d'une fusée.

- **Masque fixe par particule** : on choisit un ingénieur pour une trajectoire entière. S'il pense
  que le moteur pousse 5 % trop fort, cette hypothèse reste vraie pendant tout le vol. Son erreur
  s'accumule, ce qui est réaliste.
- **Masque rééchantillonné à chaque pas** : on change d'ingénieur toutes les secondes. Une seconde,
  l'ingénieur surestime le moteur ; la seconde suivante, un autre le sous-estime. Les erreurs
  s'annulent artificiellement, et la trajectoire paraît plus certaine qu'elle ne l'est vraiment.

Deep PILCO choisit donc la première option : **un masque par particule, gardé fixe sur tout
l'horizon**.

---

### Résumé

- **Masque fixe par particule**  
  Chaque particule suit une dynamique plausible cohérente. Les biais du modèle s'accumulent comme
  dans un vrai rollout. L'incertitude trajectorielle est mieux estimée.

- **Masque rééchantillonné à chaque pas**  
  La particule change de dynamique à chaque instant. Les erreurs deviennent indépendantes et se
  moyennent. L'incertitude est sous-estimée.

Cette distinction est l'une des raisons pour lesquelles Deep PILCO ne se contente pas de faire du
dropout naïf. Il faut propager l'incertitude **corrélée dans le temps**, pas seulement injecter du
bruit indépendant à chaque pas.

Démonstration directe :

In [ ]:
# Masque fixe vs rééchantillonné : effet sur l'incertitude propagée
# Modèle jouet : x' = 0.97 x + biais. "biais par particule" = l'analogue d'un masque de dropout.
K, H = 400, 25
def rollout(fixed_bias):
    x = np.zeros(K); bias = np.random.normal(0, 0.15, size=K)   # un "réseau échantillonné" par particule
    stds = []
    for t in range(H):
        b = bias if fixed_bias else np.random.normal(0, 0.15, size=K)  # fixe vs rééchantillonné
        x = 0.97 * x + b
        stds.append(x.std())
    return stds

plt.figure(figsize=(8, 3.6))
plt.plot(rollout(True), "o-", ms=3, color="darkred", label="masque FIXE par particule (correct)")
plt.plot(rollout(False), "s-", ms=3, color="steelblue", label="masque rééchantillonné (sous-estime)")
plt.xlabel("pas de temps"); plt.ylabel("écart-type du nuage"); plt.title("Erreur de modèle corrélée dans le temps")
plt.legend(); plt.show()

**Lecture.** Avec un masque **fixe** (rouge), l'incertitude grandit et s'accumule de façon
réaliste : chaque particule subit un biais cohérent qui s'intègre sur l'horizon. Avec un masque
**rééchantillonné** (bleu), les biais successifs s'annulent et l'écart-type reste artificiellement
petit — on se croirait bien plus certain qu'on ne l'est. Pour la planification, sous-estimer
l'incertitude est dangereux : on bâtirait des plans trop optimistes. D'où la règle d'or :
**un masque par particule, figé sur tout l'horizon.**

## III.3 — Garder le nuage gaussien : le resample par moment matching

Après quelques pas de propagation, le nuage de particules peut devenir difficile à manipuler :
étiré, asymétrique, avec plusieurs amas, ou avec quelques particules très loin des autres. C'est
normal : chaque particule suit une trajectoire différente, avec son propre état initial, sa propre
action, et son propre modèle échantillonné par dropout.

Mais si on laisse ce nuage évoluer librement pendant tout l'horizon, deux problèmes apparaissent.

D'abord, la représentation devient instable : quelques particules extrêmes peuvent dominer le coût
ou le gradient. Ensuite, on s'éloigne de l'esprit de PILCO, qui consiste à maintenir une croyance
simple sur l'état, généralement résumée par une moyenne et une covariance.

Deep PILCO utilise donc une opération de **re-résumé** à chaque pas. On prend le nuage obtenu après
propagation, puis on le remplace par une gaussienne qui a les mêmes deux premiers moments :

- même moyenne ;
- même covariance.

C'est le **moment matching**, mais fait numériquement.

Pour un nuage de particules

$$
\{x^{(1)}, x^{(2)}, \dots, x^{(K)}\},
$$

on calcule d'abord la moyenne empirique :

$$
\mu
=
\frac{1}{K}
\sum_{k=1}^{K}
x^{(k)}.
$$

Puis la covariance empirique :

$$
\Sigma
=
\frac{1}{K-1}
\sum_{k=1}^{K}
(x^{(k)}-\mu)(x^{(k)}-\mu)^\top.
$$

Ces deux quantités définissent une gaussienne :

$$
q(x) = \mathcal N(\mu, \Sigma).
$$

On rééchantillonne ensuite $K$ nouvelles particules depuis cette gaussienne :

$$
\varepsilon^{(k)} \sim \mathcal N(0,I),
\qquad
x^{(k)} \leftarrow \mu + L\varepsilon^{(k)},
$$

où $L$ est une racine de la covariance, typiquement la décomposition de Cholesky :

$$
LL^\top = \Sigma.
$$

Cette opération remplace donc un nuage potentiellement compliqué par un nuage plus régulier, mais
qui conserve les informations principales : où se trouve la masse de probabilité et dans quelles
directions elle est incertaine.



### Pourquoi faire ça ?

On peut voir les particules comme une foule de coureurs. Après quelques pas, certains partent sur
les côtés, certains accélèrent, d'autres restent groupés. Si on veut continuer à décrire la foule
simplement, on peut retenir deux informations :

- le centre de la foule ;
- son étalement.

Puis on recrée une foule propre avec le même centre et le même étalement. On perd les détails fins,
mais on garde une description stable.

C'est exactement ce que fait le resample par moment matching.



### Ce qu'on gagne

Cette étape apporte trois bénéfices.

1. **Stabilité numérique.**  
   Le nuage ne dégénère pas en quelques particules extrêmes ou en un amas trop concentré.

2. **Continuité avec PILCO.**  
   Le PILCO original propageait une gaussienne en calculant analytiquement ses moments. Deep PILCO
   fait la même chose, mais les moments sont estimés par Monte-Carlo.

3. **Différentiabilité.**  
   Le coût final dépend de $\mu$ et $\Sigma$, qui dépendent eux-mêmes des particules, donc du modèle
   et de la politique. On peut donc rétropropager le gradient jusqu'aux paramètres de la politique.



### Ce qu'on perd

Le prix à payer est une approximation. Si le vrai nuage devient multimodal, par exemple avec deux
amas très séparés, une seule gaussienne ne peut pas représenter correctement cette structure. Elle
va placer une masse artificielle entre les deux modes.

Donc le resample par moment matching est un compromis :

$$
\text{distribution réelle compliquée}
\quad\longrightarrow\quad
\mathcal N(\mu,\Sigma)
$$

On perd les détails, mais on garde une représentation simple, stable et optimisable.

C'est l'exact pendant du moment matching analytique du notebook 10, mais réalisé **numériquement** :
on remplace la distribution réelle par une gaussienne qui possède les mêmes deux premiers moments.
Cette opération garde le calcul tractable sur tout l'horizon et permet d'optimiser la politique par
rétropropagation.

In [ ]:
def propagate_particles(net, policy, particles, masks, target, W):
    actions = policy(particles)                              # [K, Du]
    delta = net(torch.cat([particles, actions], -1), masks)  # masques FIXES de la particule
    nxt = particles + delta                                  # x' = x + Δx
    cost = saturating_cost(nxt, target, W).mean()            # coût moyen sur le nuage
    mu = nxt.mean(0); cen = nxt - mu                         # resample par moment matching
    cov = cen.t() @ cen / (nxt.shape[0] - 1) + 1e-5 * torch.eye(nxt.shape[1])
    resampled = mu + torch.randn_like(nxt) @ torch.linalg.cholesky(cov).t()
    return resampled, cost

def saturating_cost(x, target, W):
    d = x - target
    return 1.0 - torch.exp(-0.5 * (d @ W * d).sum(-1))

# Visualisation : un nuage de particules propagé puis re-résumé en gaussienne
torch.manual_seed(0)
dummy_net = BayesianDynamicsNet(2, 1, hidden=32, p=0.1)      # 1D : état scalaire + action factice
class _Zero(nn.Module):
    def forward(self, x): return torch.zeros(x.shape[0], 1)
parts = torch.randn(300, 1) * 0.3 + 1.0
masks = dummy_net.sample_masks(300)
# un pas : Δx via le réseau (entrée [x, 0]) -> nuage déformé -> resample gaussien
with torch.no_grad():
    delta = dummy_net(torch.cat([parts, torch.zeros(300, 1)], -1), masks)
    deformed = parts + delta
    mu = deformed.mean(0); cov = ((deformed - mu).t() @ (deformed - mu) / 299 + 1e-5)
    resamp = mu + torch.randn(300, 1) @ torch.linalg.cholesky(cov.reshape(1, 1)).t()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharex=True)
axes[0].hist(deformed[:, 0].numpy(), bins=30, color="steelblue", alpha=0.7)
axes[0].set_title("Nuage propagé (peut être biscornu)")
axes[1].hist(resamp[:, 0].numpy(), bins=30, color="darkorange", alpha=0.7)
axes[1].set_title("Après resample $\\mathcal{N}(\\mu, \\Sigma)$"); plt.tight_layout(); plt.show()

**Lecture.** À gauche, le nuage après un pas de réseau ; à droite, sa version
**re-gaussianisée** (même moyenne, même variance). Le resample « nettoie » la distribution à chaque
étape pour qu'elle reste une gaussienne maniable, exactement comme le moment matching analytique le
faisait pour le GP — sauf qu'ici tout est numérique. C'est ce qui permet d'enchaîner les pas sans
que la représentation n'explose en complexité.

In [ ]:
def predict_trajectory_particles(net, policy, mu0, sigma0, horizon, target, W, K=30, project_fn=None):
    # Rollout par particules ; masques FIXES sur tout l'horizon ; coût total différentiable.
    L0 = torch.linalg.cholesky(sigma0 + 1e-5 * torch.eye(mu0.shape[0]))
    particles = mu0 + torch.randn(K, mu0.shape[0]) @ L0.t()
    masks = net.sample_masks(K)                              # tirés une fois, gardés tout l'horizon
    total = torch.zeros(())
    for _ in range(horizon):
        particles, c = propagate_particles(net, policy, particles, masks, target, W)
        if project_fn is not None:
            particles = project_fn(particles)
        total = total + c
    return total

# Mini-test : coût total fini et différentiable par rapport à une politique 5D.
class MLPPolicy(nn.Module):
    def __init__(self, sd, ad, hidden=32, action_high=2.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(sd, hidden), nn.Tanh(), nn.Linear(hidden, ad))
        self.action_high = action_high
    def forward(self, x):
        return self.action_high * torch.tanh(self.net(x))

net5 = BayesianDynamicsNet(6, 5, hidden=32, p=0.1)
pol = MLPPolicy(5, 1, action_high=3.0)
mu0 = torch.tensor([0., 0., 1., 0., 0.])
S0 = 0.02 * torch.eye(5)
target = torch.tensor([0., 0., 1., 0., 0.])
W = torch.eye(5)
J = predict_trajectory_particles(net5, pol, mu0, S0, horizon=10, target=target, W=W, K=20)
g = torch.autograd.grad(J, next(pol.parameters()))[0]
assert torch.isfinite(J) and torch.isfinite(g).all()
print(f"✓ rollout par particules OK : J = {float(J):.3f}, gradient fini -> optimisable par Adam")

---
# Partie IV — GP vs BNN, et la boucle Deep PILCO

## IV.1 — Les deux modèles côte à côte

| Aspect | GP (PILCO) | BNN MC-dropout (Deep PILCO) |
|--------|-----------|------------------------------|
| Modèle | Gaussian Process | réseau de neurones bayésien |
| Incertitude de sortie | **exacte**, naturelle | approchée (dropout + MC) |
| Propagation d'incertitude | **analytique** (moment matching) | **par particules** |
| Corrélation temporelle | gérée par les équations | gérée par les **masques fixes** |
| Coût en données | $O(n^3)$ — petit $n$ seulement | passe à l'échelle |
| Haute dimension | difficile | réaliste |
| Tractabilité analytique | très forte | plus faible |
| Politique | doit être **intégrable** (RBF) | **quelconque** (MLP) — on optimise par autograd |

Le compromis se résume ainsi : **PILCO** est plus élégant et exact mais ne passe pas à l'échelle ;
**Deep PILCO** est approché mais flexible et scalable. Notez la conséquence libératrice de la
dernière ligne : comme on n'a plus besoin de propagation analytique, la politique n'a plus besoin
d'être un RBF intégrable — **un simple MLP** suffit (on l'optimise par rétropropagation à travers
les particules).

In [ ]:
# GP vs BNN sur le même problème jouet : incertitude comparée
def tiny_gp(Xtr, ytr, Xs, ell=0.7, sf=1.0, sn=0.1):
    def k(A, B):
        d = (A[:, None, 0] - B[None, :, 0]) / ell; return sf * torch.exp(-0.5 * d ** 2)
    Kmat = k(Xtr, Xtr) + sn ** 2 * torch.eye(len(Xtr))
    L = torch.linalg.cholesky(Kmat); alpha = torch.cholesky_solve(ytr[:, None], L)[:, 0]
    Ks = k(Xtr, Xs); mean = Ks.t() @ alpha
    v = torch.cholesky_solve(Ks, L); var = (sf - (Ks * v).sum(0)).clamp_min(0)
    return mean, var

with torch.no_grad():
    gm, gv = tiny_gp(Xtr, ytr.squeeze(1), grid)
    bp = torch.stack([bnn(grid)[:, 0] for _ in range(100)]); bm, bs = bp.mean(0), bp.std(0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
axes[0].plot(grid[:, 0], gm, "b"); axes[0].fill_between(grid[:, 0], gm - 2 * gv.sqrt(), gm + 2 * gv.sqrt(), alpha=0.25, color="steelblue")
axes[0].scatter(Xtr[:, 0], ytr[:, 0], c="red", s=12); axes[0].set_title("GP (incertitude exacte, lisse)")
axes[1].plot(grid[:, 0], bm, "b"); axes[1].fill_between(grid[:, 0], bm - 2 * bs, bm + 2 * bs, alpha=0.25, color="orange")
axes[1].scatter(Xtr[:, 0], ytr[:, 0], c="red", s=12); axes[1].set_title("BNN MC-dropout (approchée, scalable)")
plt.tight_layout(); plt.show()

**Lecture.** Les deux modèles racontent la même histoire — certains près des données,
incertains loin — mais autrement. Le GP (gauche) donne une bande **lisse et calibrée** ; le BNN
(droite) donne une incertitude **approchée**, parfois un peu irrégulière, mais obtenue avec un
réseau qui passera à l'échelle là où le GP cale. **Quand choisir lequel ?** GP pour les petits
systèmes basse dimension où l'exactitude prime ; BNN dès que les données ou les dimensions
abondent. Assemblons maintenant la boucle Deep PILCO.

## IV.2 — La boucle Deep PILCO sur `InvertedPendulum-v5`

La boucle externe reste celle de PILCO : **entraîner le modèle**, **imaginer**, **ajuster la
politique**, puis **collecter une seule trajectoire réelle**. Ce qui change est la recette
intérieure. Avec un GP, PILCO pouvait propager des moments analytiquement ; ici, tout passe par un
BNN, des particules et des gradients stochastiques. Les détails de protocole comptent donc beaucoup.

### Algorithme complet

$$
\boxed{
\begin{aligned}
&\textbf{Initialiser } \pi_\theta,\ f_\phi,\ \mathcal{D}\leftarrow\varnothing \\
&\textbf{Choisir des seeds de validation fixes } \mathcal{S}_{val}
\textbf{ et des seeds held-out } \mathcal{S}_{test} \\[1mm]
&\textbf{Collecter } N_{warmup} \textbf{ rollouts aléatoires officiels :} \\
&\quad \mathcal{D}\leftarrow
\mathcal{D}\cup\{(x_t,u_t,\Delta x_t=x_{t+1}-x_t)\} \\[2mm]
&\textbf{Évaluer la politique initiale sur } \mathcal{S}_{val}
\textbf{ et sauvegarder } \pi_{\theta}^{best}\leftarrow\pi_\theta \\[2mm]
&\textbf{Pour } i=1,\dots,N_{iter} \textbf{ faire :} \\[1mm]
&\quad \textcolor{gray}{\triangleright\ \text{1. Apprendre la dynamique locale}} \\
&\quad
\phi \leftarrow
\operatorname{Adam}\!\left[
\min_\phi
\frac{1}{|\mathcal{B}|}
\sum_{(x,u,\Delta x)\in\mathcal{B}}
\left\|f_\phi(x,u)-\Delta x\right\|^2
\right]
\quad \text{pendant } N_{model} \text{ updates} \\[2mm]
&\quad \textcolor{gray}{\triangleright\ \text{2. Imaginer avec particules et masques fixes}} \\
&\quad \textbf{pour chaque update politique :} \\
&\quad\quad x_0^{(k)}\sim p_0(x),
\qquad
m^{(k)}\sim \operatorname{Bernoulli}(1-p)
\quad \text{pour } k=1,\dots,K \\[1mm]
&\quad\quad \textbf{pour } t=0,\dots,H-1 \textbf{ faire :} \\
&\quad\quad\quad u_t^{(k)}=\pi_\theta(x_t^{(k)}) \\
&\quad\quad\quad
\Delta \hat x_t^{(k)}
=
f_{\phi,m^{(k)}}\!\left(x_t^{(k)},u_t^{(k)}\right) \\
&\quad\quad\quad
\hat x_{t+1}^{(k)}
=
\operatorname{project}\!\left(x_t^{(k)}+\Delta \hat x_t^{(k)}\right) \\
&\quad\quad\quad
\hat x_{t+1}^{(1:K)}
\leftarrow
\operatorname{resample}\!\left(
\mu(\hat x_{t+1}^{(1:K)}),
\Sigma(\hat x_{t+1}^{(1:K)})
\right) \\
&\quad\quad \textbf{fin pour} \\[1mm]
&\quad\quad
J(\theta)
=
\sum_{t=0}^{H-1}
\frac{1}{K}
\sum_{k=1}^{K}
c\!\left(x_t^{(k)},u_t^{(k)}\right) \\[1mm]
&\quad\quad
\theta \leftarrow
\theta-\alpha_\pi\nabla_\theta J(\theta)
\quad \text{si l'update reste finie et acceptable} \\[2mm]
&\quad \textcolor{gray}{\triangleright\ \text{3. Retour au vrai environnement}} \\
&\quad \tau_i \leftarrow
\operatorname{rollout}_{env}
\!\left(\pi_\theta + \varepsilon_i\right) \\
&\quad
\mathcal{D}\leftarrow
\operatorname{cap}\!\left(
\mathcal{D}\cup
\{(x_t,u_t,x_{t+1}-x_t)\}_{\tau_i}
\right) \\[2mm]
&\quad \textcolor{gray}{\triangleright\ \text{4. Sélection sur validation réelle}} \\
&\quad \textbf{si } i=1
\textbf{ ou } i \equiv 0 \pmod{E}
\textbf{ ou } i=N_{iter} : \\
&\quad\quad
R_{val}(\pi_\theta)
\leftarrow
\operatorname{Eval}_{env}(\pi_\theta,\mathcal{S}_{val}) \\
&\quad\quad
\textbf{si } R_{val}(\pi_\theta)>R_{val}(\pi_{\theta}^{best})
\textbf{ alors }
\pi_{\theta}^{best}\leftarrow\pi_\theta \\
&\textbf{fin pour} \\[2mm]
&\textbf{Restaurer } \pi_{\theta}^{best}
\textbf{ puis mesurer une seule fois sur } \mathcal{S}_{test}.
\end{aligned}
}
$$

### Lecture de l'algorithme

Les quatre blocs correspondent aux quatre lignes importantes du code :

- `agent.fit_dynamics(...)` entraîne le BNN sur le dataset cumulatif
  $\mathcal{D}=\{(x,u,\Delta x)\}$ ;
- `agent.optimize_policy(...)` calcule un coût imaginé $J(\theta)$ en déroulant des particules avec
  **masques de dropout fixes** ;
- `agent.collect_rollout(...)` revient au simulateur réel pour ajouter une trajectoire naturelle ;
- `agent.keep_best_real_policy(...)` sélectionne la politique sur des seeds fixes, puis `restore_best`
  empêche de finir avec un dernier update moins bon.

Le point subtil est que l'optimisation de politique ne fait pas confiance aveuglément au coût imaginé.
Le coût imaginé fournit le gradient, mais la **sélection** se fait sur l'environnement réel. C'est
important parce qu'un BNN peut devenir localement trop optimiste : une baisse de $J(\theta)$ dans les
particules n'implique pas automatiquement une meilleure trajectoire MuJoCo.

### La recette qui a tenu

1. **Politique MLP.** Contrairement au PILCO analytique, Deep PILCO n'a plus besoin d'une politique
   intégrable en forme close. On utilise donc un **MLP** simple, plus expressif pour le contrôle local
   des particules.
2. **Coût dense + barrière.** Le signal d'optimisation vient d'un coût quadratique dense
   `ip_stabilization_cost`, avec un poids fort sur $\theta^2$, un petit coût d'action, et une
   **barrière** qui se déclenche avant la vraie chute. Ici, contrairement au PILCO classique, on ne
   veut pas d'un coût saturant presque plat loin de l'upright : il faut un gradient partout.
3. **Dropout calibré.** Le BNN utilise un **dropout de $0.08$**, et surtout des **masques fixes par
   particule** sur tout l'horizon. Chaque particule suit une dynamique plausible cohérente, pas un
   monde qui change de lois physiques à chaque pas.
4. **Propagation.** On propage **96 particules** sur un horizon imaginé de **80 pas**, puis on résume
   le nuage par **moment matching empirique** avant de rééchantillonner.
5. **Budgets.** Chaque itération externe lance **260 updates modèle** et **70 updates politique**.
   C'est le compromis qui a tenu dans ce notebook : assez long pour faire converger le BNN local,
   assez court pour rester exécutable.
6. **Warmup naturel.** On commence par **8 rollouts aléatoires** de l'environnement réel. Ce n'est pas
   une initialisation artificielle du reset : le BNN voit la distribution officielle du monde telle
   qu'elle est, avec ses petites perturbations autour de l'upright.
7. **Sélection honnête.** La politique est sélectionnée sur des **seeds fixes** ; on **restore** la
   meilleure politique réelle observée, au lieu de supposer que la baisse du coût imaginé garantit
   automatiquement une hausse du score réel.
8. **Held-out final.** Le chiffre à retenir vient de **20 seeds held-out disjointes**, jamais utilisées
   pour la sélection. C'est cette mesure finale qui a donné **$1000 \pm 0$**.

### Pourquoi le score `1000` n'est pas directement comparable au papier

Le succès ici est spectaculaire **dans Gymnasium**, mais il ne faut pas le comparer naïvement au
papier Deep PILCO. `InvertedPendulum-v5` reset déjà près de la verticale : la tâche est une
**stabilisation locale très robuste**, pas un **swing-up complet** depuis le bas. Le score plafond
de `1000` signifie « rester vivant 1000 pas », ce qui est sévère en durée mais plus local en
géométrie qu'une tâche de swing-up custom.

### Comparaison précise avec Deep PILCO (2016)

| Élément | Deep PILCO (papier) | Ce notebook pédagogique |
|---|---|---|
| Tâche | **swing-up custom** | `InvertedPendulum-v5`, upright reset officiel |
| Politique | **RBF 50 centres** | **MLP** local pour particules |
| Dynamique | BNN **2×200 sigmoid**, dropout **0.05** | BNN plus compact, dropout **0.08** |
| Optimisation interne | **5000** updates modèle, **1000** updates politique | **260 / 70** |
| Itérations externes | **100** | mini-training court de notebook |
| Données d'entraînement | **10 trajectoires récentes** | buffer pédagogique capé, warmup naturel puis 1 rollout réel par itération |
| Propagation | particules + masques fixes | idem, avec **96 particules** et horizon **80** |

Autrement dit : on garde l'**idée algorithmique** de Deep PILCO, mais on n'essaie pas de mimer à
l'identique le budget ni la tâche du papier. Le notebook est une **variante pédagogique moderne**
dont la question honnête est : *la recette apprend-elle réellement sur Gymnasium, avec un protocole
de sélection propre ?*

In [ ]:
# Configuration complète du mini-training Deep PILCO.
ENV_ID = "InvertedPendulum-v5"
DEEP_PILCO_SEED = 0
DEEP_PILCO_ITERATIONS = 22
INIT_RANDOM_ROLLOUTS = 8
COLLECT_HORIZON = 120
EVAL_HORIZON = 1000
MODEL_UPDATES = 260
POLICY_UPDATES = 70
MODEL_BATCH_SIZE = 128
N_PARTICLES = 96
IMAGINE_HORIZON = 80
BUFFER_CAP = 2500
EVAL_EVERY = 3
EVAL_EPISODES = 20
VALIDATION_SEEDS = tuple(range(400, 400 + EVAL_EPISODES))
HELDOUT_SEEDS = tuple(range(900, 900 + EVAL_EPISODES))
EXPLORATION_NOISE_START = 0.35

np.random.seed(DEEP_PILCO_SEED)
torch.manual_seed(DEEP_PILCO_SEED)


In [ ]:
def clone_params(module):
    return [p.detach().clone() for p in module.parameters()]

def restore_params(module, backup):
    with torch.no_grad():
        for param, saved in zip(module.parameters(), backup):
            param.copy_(saved)

def encode_ip_obs(obs):
    # Observation brute Gymnasium : [x, theta, x_dot, theta_dot].
    x, theta, x_dot, theta_dot = np.asarray(obs, dtype=np.float32)
    return np.asarray([x, np.sin(theta), np.cos(theta), x_dot, theta_dot], dtype=np.float32)

def project_ip_encoded_torch(x):
    # Le BNN vit dans [x, sin(theta), cos(theta), x_dot, theta_dot].
    angle = x[..., 1:3]
    angle = angle / torch.linalg.norm(angle, dim=-1, keepdim=True).clamp_min(1e-6)
    return torch.cat([x[..., :1], angle, x[..., 3:]], dim=-1)

def ip_angle_deg_from_encoded(encoded):
    return float(np.degrees(np.arctan2(encoded[1], encoded[2])))

def deep_policy_action(policy, encoded_obs, noise_std=0.0, rng=None):
    with torch.no_grad():
        action = policy(torch.tensor(encoded_obs, dtype=torch.float32)).detach().cpu().numpy()
    if noise_std > 0.0:
        assert rng is not None
        action = action + rng.normal(0.0, noise_std, size=action.shape)
    return np.clip(action, -3.0, 3.0).astype(np.float32)

def rollout_deep_policy(env, policy, *, seed, random=False, noise_std=0.0, max_steps=1000):
    obs, _ = env.reset(seed=seed)
    env.action_space.seed(seed)
    encoded = encode_ip_obs(obs)
    rng = np.random.default_rng(seed)
    X, Y, rewards, angles, actions, positions = [], [], [], [], [], []

    for _ in range(max_steps):
        if random:
            action = env.action_space.sample().astype(np.float32)
        else:
            action = deep_policy_action(policy, encoded, noise_std=noise_std, rng=rng)

        next_obs, reward, terminated, truncated, _ = env.step(action)
        next_encoded = encode_ip_obs(next_obs)
        X.append(np.concatenate([encoded, action]).astype(np.float32))
        Y.append((next_encoded - encoded).astype(np.float32))
        rewards.append(float(reward))
        angles.append(ip_angle_deg_from_encoded(next_encoded))
        actions.append(float(action[0]))
        positions.append(float(next_encoded[0]))
        encoded = next_encoded
        if terminated or truncated:
            break

    return {
        "X": torch.tensor(np.asarray(X), dtype=torch.float32),
        "Y": torch.tensor(np.asarray(Y), dtype=torch.float32),
        "reward": float(np.sum(rewards)),
        "length": len(rewards),
        "angles": np.asarray(angles, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.float32),
        "positions": np.asarray(positions, dtype=np.float32),
        "mean_abs_angle": float(np.mean(np.abs(angles))) if angles else float("nan"),
        "max_abs_angle": float(np.max(np.abs(angles))) if angles else float("nan"),
        "upright_fraction_5deg": float(np.mean(np.abs(angles) <= 5.0)) if angles else 0.0,
        "mean_abs_action": float(np.mean(np.abs(actions))) if actions else float("nan"),
    }


In [ ]:
def summarize_rollouts(rollouts):
    returns = np.asarray([r["reward"] for r in rollouts], dtype=np.float32)
    lengths = np.asarray([r["length"] for r in rollouts], dtype=np.float32)
    angles = np.asarray([r["mean_abs_angle"] for r in rollouts], dtype=np.float32)
    max_angles = np.asarray([r["max_abs_angle"] for r in rollouts], dtype=np.float32)
    upright = np.asarray([r["upright_fraction_5deg"] for r in rollouts], dtype=np.float32)
    return {
        "mean_reward": float(np.mean(returns)),
        "std_reward": float(np.std(returns)),
        "mean_length": float(np.mean(lengths)),
        "mean_abs_angle": float(np.mean(angles)),
        "mean_max_abs_angle": float(np.mean(max_angles)),
        "upright_fraction_5deg": float(np.mean(upright)),
    }

def evaluate_deep_policy(policy, *, seeds, max_steps=EVAL_HORIZON):
    env_eval = gym.make(ENV_ID)
    try:
        rollouts = [rollout_deep_policy(env_eval, policy, seed=seed, max_steps=max_steps) for seed in seeds]
    finally:
        env_eval.close()
    return summarize_rollouts(rollouts)

def evaluate_random_deep(*, seeds, max_steps=EVAL_HORIZON):
    env_eval = gym.make(ENV_ID)
    try:
        dummy = MLPPolicy(5, 1, action_high=3.0)
        rollouts = [rollout_deep_policy(env_eval, dummy, seed=seed, random=True, max_steps=max_steps) for seed in seeds]
    finally:
        env_eval.close()
    return summarize_rollouts(rollouts)

def evaluate_zero_action_deep(*, seeds, max_steps=EVAL_HORIZON):
    env_eval = gym.make(ENV_ID)
    rollouts = []
    try:
        for seed in seeds:
            obs, _ = env_eval.reset(seed=seed)
            env_eval.action_space.seed(seed)
            encoded = encode_ip_obs(obs)
            rewards, angles, actions, positions = [], [], [], []
            for _ in range(max_steps):
                action = np.zeros(env_eval.action_space.shape, dtype=np.float32)
                next_obs, reward, terminated, truncated, _ = env_eval.step(action)
                encoded = encode_ip_obs(next_obs)
                rewards.append(float(reward))
                angles.append(ip_angle_deg_from_encoded(encoded))
                actions.append(0.0)
                positions.append(float(encoded[0]))
                if terminated or truncated:
                    break
            rollouts.append({
                "reward": float(np.sum(rewards)),
                "length": len(rewards),
                "mean_abs_angle": float(np.mean(np.abs(angles))) if angles else float("nan"),
                "max_abs_angle": float(np.max(np.abs(angles))) if angles else float("nan"),
                "upright_fraction_5deg": float(np.mean(np.abs(angles) <= 5.0)) if angles else 0.0,
                "mean_abs_action": 0.0,
            })
    finally:
        env_eval.close()
    return summarize_rollouts(rollouts)


In [ ]:
def cap_dataset(X, Y, cap):
    return (X, Y) if len(X) <= cap else (X[-cap:], Y[-cap:])

def train_bnn_minibatches(net, X, Y, *, updates, batch_size, lr, seed):
    X, Y = cap_dataset(X, Y, BUFFER_CAP)
    n = len(X)
    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=gen)
    split = max(1, int(0.8 * n))
    train_idx = perm[:split]
    val_idx = perm[split:] if split < n else perm[:split]
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-5)
    losses = []
    net.train()
    for _ in range(updates):
        idx = train_idx[torch.randint(0, len(train_idx), (min(batch_size, len(train_idx)),), generator=gen)]
        pred = net(torch.cat([X[idx]], dim=-1))
        loss = F.mse_loss(pred, Y[idx])
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 10.0)
        opt.step()
        losses.append(float(loss.detach()))
    with torch.no_grad():
        train_loss = float(F.mse_loss(net(X[train_idx]), Y[train_idx]).detach())
        val_loss = float(F.mse_loss(net(X[val_idx]), Y[val_idx]).detach())
    return float(np.mean(losses[-25:])), train_loss, val_loss, len(X)


In [ ]:
def ip_stabilization_cost(x, actions):
    # Coût dense : rester proche de x=0, theta=0, vitesses faibles et action modérée.
    theta = torch.atan2(x[:, 1], x[:, 2])
    x_pos = x[:, 0]
    x_dot = x[:, 3]
    theta_dot = x[:, 4]
    torque = actions.squeeze(-1).clamp(-3.0, 3.0)
    near_failure = torch.relu(theta.abs() - 0.14) ** 2
    return (
        0.5 * x_pos.square()
        + 35.0 * theta.square()
        + 0.05 * x_dot.square()
        + 0.7 * theta_dot.square()
        + 0.01 * torque.square()
        + 150.0 * near_failure
    )

def rollout_cost_particles(net, policy, particles, masks, horizon):
    total = torch.zeros((), dtype=particles.dtype)
    x = particles
    for _ in range(horizon):
        actions = policy(x).clamp(-3.0, 3.0)
        delta = net(torch.cat([x, actions], dim=-1), masks)
        x_next = project_ip_encoded_torch(x + delta)
        total = total + ip_stabilization_cost(x_next, actions).mean()
        # Moment matching numérique : stabilise le nuage comme dans Deep PILCO.
        mu = x_next.mean(dim=0)
        centered = x_next - mu
        cov = centered.t() @ centered / max(x_next.shape[0] - 1, 1)
        cov = cov + 1e-5 * torch.eye(cov.shape[0], dtype=cov.dtype)
        eps = torch.randn_like(x_next)
        x = project_ip_encoded_torch(mu + eps @ torch.linalg.cholesky(cov).t())
    return total / horizon

def sample_reset_particles(n, seed):
    env_reset = gym.make(ENV_ID)
    samples = []
    try:
        for k in range(n):
            obs, _ = env_reset.reset(seed=seed + k)
            samples.append(encode_ip_obs(obs))
    finally:
        env_reset.close()
    return torch.tensor(np.asarray(samples), dtype=torch.float32)

def optimize_deep_policy(net, policy, *, steps, lr, seed):
    net.eval()
    backup = clone_params(policy)
    best = clone_params(policy)
    best_cost = float("inf")
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    base_particles = sample_reset_particles(N_PARTICLES, seed)
    masks = net.sample_masks(N_PARTICLES)

    with torch.no_grad():
        before = float(rollout_cost_particles(net, policy, base_particles, masks, IMAGINE_HORIZON))

    accepted = 0
    for _ in range(steps):
        opt.zero_grad()
        cost = rollout_cost_particles(net, policy, base_particles, masks, IMAGINE_HORIZON)
        if not torch.isfinite(cost):
            restore_params(policy, backup)
            break
        cost.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 5.0)
        opt.step()
        value = float(cost.detach())
        if np.isfinite(value) and value < best_cost:
            best_cost = value
            best = clone_params(policy)
            accepted += 1

    restore_params(policy, best if np.isfinite(best_cost) else backup)
    with torch.no_grad():
        after = float(rollout_cost_particles(net, policy, base_particles, masks, IMAGINE_HORIZON))
    return before, after, accepted


## IV.2 bis — Assembler Deep PILCO

Le BNN et la politique ont été construits séparément ; `DeepPILCOAgent` les réunit avec les sauvegardes
de politique utilisées par l'évaluation réelle. La boucle externe garde les données et l'environnement,
et continue à raconter les quatre étapes : ajuster le BNN, optimiser dans les particules, collecter une
trajectoire réelle, puis évaluer sur des seeds fixes.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    env["environnement réel"] --> data["transitions cumulées"]
    data --> bnn["BNN de dynamique"]
    bnn --> particles["particules + masques fixes"]
    particles --> policy["optimisation de la politique"]
    policy --> env
    class bnn,particles primary
    class policy secondary
```

In [ ]:
class DeepPILCOAgent:
    def __init__(self, policy, dynamics, initial_eval_reward):
        self.policy = policy
        self.dynamics = dynamics
        self.initial_policy = clone_params(policy)
        self.best_policy = clone_params(policy)
        self.best_eval_reward = float(initial_eval_reward)
        self.best_eval_label = "initiale"

    def fit_dynamics(self, data_X, data_Y, *, seed):
        return train_bnn_minibatches(
            self.dynamics, data_X, data_Y,
            updates=MODEL_UPDATES,
            batch_size=MODEL_BATCH_SIZE,
            lr=8e-4,
            seed=seed,
        )

    def optimize_policy(self, *, seed):
        return optimize_deep_policy(
            self.dynamics, self.policy,
            steps=POLICY_UPDATES, lr=3e-3, seed=seed,
        )

    def collect_rollout(self, env, **kwargs):
        return rollout_deep_policy(env, self.policy, **kwargs)

    def evaluate(self, seeds):
        return evaluate_deep_policy(self.policy, seeds=seeds)

    def keep_best_real_policy(self, evaluation, label):
        if evaluation["mean_reward"] > self.best_eval_reward:
            self.best_eval_reward = evaluation["mean_reward"]
            self.best_policy = clone_params(self.policy)
            self.best_eval_label = label

    def restore_initial(self):
        restore_params(self.policy, self.initial_policy)

    def restore_best(self):
        restore_params(self.policy, self.best_policy)


In [ ]:
torch.manual_seed(DEEP_PILCO_SEED)
np.random.seed(DEEP_PILCO_SEED)

env = gym.make(ENV_ID)
env.action_space.seed(DEEP_PILCO_SEED)
policy = MLPPolicy(5, 1, hidden=64, action_high=3.0)
net = BayesianDynamicsNet(6, 5, hidden=96, p=0.08)

random_eval = evaluate_random_deep(seeds=VALIDATION_SEEDS)
zero_eval = evaluate_zero_action_deep(seeds=VALIDATION_SEEDS)
initial_eval = evaluate_deep_policy(policy, seeds=VALIDATION_SEEDS)
print(
    f"Baseline random reset officiel : reward={random_eval['mean_reward']:7.1f} ± {random_eval['std_reward']:.1f} | "
    f"longueur={random_eval['mean_length']:.1f} | |theta|={random_eval['mean_abs_angle']:.2f}°"
)
print(
    f"Baseline zéro-action reset officiel : reward={zero_eval['mean_reward']:7.1f} ± {zero_eval['std_reward']:.1f} | "
    f"longueur={zero_eval['mean_length']:.1f} | |theta|={zero_eval['mean_abs_angle']:.2f}°"
)
print(
    f"Politique initiale reset officiel : reward={initial_eval['mean_reward']:7.1f} ± {initial_eval['std_reward']:.1f} | "
    f"longueur={initial_eval['mean_length']:.1f} | |theta|={initial_eval['mean_abs_angle']:.2f}°"
)

warmup = [rollout_deep_policy(env, policy, seed=10 + ep, random=True, max_steps=COLLECT_HORIZON) for ep in range(INIT_RANDOM_ROLLOUTS)]
dX = torch.cat([r["X"] for r in warmup])
dY = torch.cat([r["Y"] for r in warmup])
print(f"Transitions initiales aléatoires : {len(dX)}")

hist = {
    "reward": [], "length": [], "mean_abs_angle": [], "max_abs_angle": [],
    "upright_fraction_5deg": [], "mean_abs_action": [],
    "batch_loss": [], "model_loss": [], "val_loss": [],
    "imagined_before": [], "imagined_after": [], "policy_accepted_steps": [], "data_size": [],
    "eval_step": [], "eval_mean": [], "eval_std": [], "eval_length": [],
    "eval_abs_angle": [], "eval_max_abs_angle": [], "eval_upright_fraction_5deg": [],
}
agent = DeepPILCOAgent(policy, net, initial_eval["mean_reward"])


pbar = tqdm(range(1, DEEP_PILCO_ITERATIONS + 1), desc="Mini Deep PILCO")
for it in pbar:
    # 1. Apprendre la dynamique locale sur toutes les transitions disponibles.
    batch_loss, train_loss, val_loss, used_points = agent.fit_dynamics(
        dX, dY, seed=1000 + it,
    )

    # 2. Optimiser la politique dans le modèle avec particules et masques fixes.
    before, after, accepted_steps = agent.optimize_policy(seed=2000 + it)

    # 3. Retour au vrai simulateur : une trajectoire naturelle enrichit le dataset.
    noise = EXPLORATION_NOISE_START * max(0.0, 1.0 - (it - 1) / max(1, DEEP_PILCO_ITERATIONS - 1))
    trace = agent.collect_rollout(env, seed=100 + it, noise_std=noise, max_steps=COLLECT_HORIZON)
    dX = torch.cat([dX, trace["X"]]); dY = torch.cat([dY, trace["Y"]])
    dX, dY = cap_dataset(dX, dY, BUFFER_CAP)

    for key in ["reward", "length", "mean_abs_angle", "max_abs_angle", "upright_fraction_5deg", "mean_abs_action"]:
        hist[key].append(trace[key])
    hist["batch_loss"].append(batch_loss)
    hist["model_loss"].append(train_loss)
    hist["val_loss"].append(val_loss)
    hist["imagined_before"].append(before)
    hist["imagined_after"].append(after)
    hist["policy_accepted_steps"].append(accepted_steps)
    hist["data_size"].append(len(dX))

    if it == 1 or it % EVAL_EVERY == 0 or it == DEEP_PILCO_ITERATIONS:
        ev = agent.evaluate(VALIDATION_SEEDS)
        hist["eval_step"].append(it)
        hist["eval_mean"].append(ev["mean_reward"])
        hist["eval_std"].append(ev["std_reward"])
        hist["eval_length"].append(ev["mean_length"])
        hist["eval_abs_angle"].append(ev["mean_abs_angle"])
        hist["eval_max_abs_angle"].append(ev["mean_max_abs_angle"])
        hist["eval_upright_fraction_5deg"].append(ev["upright_fraction_5deg"])
        agent.keep_best_real_policy(ev, f"itération {it}")

    pbar.set_postfix(length=f"{trace['length']:.0f}", eval=f"{hist['eval_length'][-1]:.0f}" if hist["eval_length"] else "n/a", val=f"{val_loss:.4f}", J=f"{after:.3f}")

env.close()
heldout_seeds = HELDOUT_SEEDS
agent.restore_initial()
initial_heldout_eval = evaluate_deep_policy(policy, seeds=heldout_seeds)
agent.restore_best()
final_eval = evaluate_deep_policy(policy, seeds=heldout_seeds)
_env_trace = gym.make(ENV_ID)
try:
    final_trace = rollout_deep_policy(_env_trace, policy, seed=999, max_steps=EVAL_HORIZON)
finally:
    _env_trace.close()

print(f"Meilleure politique retenue en training : {agent.best_eval_label} (eval length={agent.best_eval_reward:.1f})")
print(
    f"Initial held-out reset officiel : longueur={initial_heldout_eval['mean_length']:7.1f} ± {initial_heldout_eval['std_reward']:.1f} | "
    f"|theta|={initial_heldout_eval['mean_abs_angle']:.2f}°"
)
print(
    f"Final held-out reset officiel   : longueur={final_eval['mean_length']:7.1f} ± {final_eval['std_reward']:.1f} | "
    f"|theta|={final_eval['mean_abs_angle']:.2f}° | upright<5°={100*final_eval['upright_fraction_5deg']:.1f}%"
)
print(
    f"Replay final reset officiel     : longueur={final_trace['length']} | "
    f"|theta|max={final_trace['max_abs_angle']:.2f}° | |action|={final_trace['mean_abs_action']:.2f}"
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

axes[0, 0].plot(hist["length"], "o-", color="darkgreen", alpha=0.85, label="collecte")
axes[0, 0].axhline(random_eval["mean_length"], color="gray", ls="--", label="random eval")
axes[0, 0].axhline(zero_eval["mean_length"], color="black", ls="-.", label="zéro-action")
axes[0, 0].axhline(initial_eval["mean_length"], color="orange", ls=":", label="initial eval")
axes[0, 0].set_title("Longueur réelle collectée")
axes[0, 0].set_xlabel("itération Deep PILCO")
axes[0, 0].set_ylabel("pas avant chute")
axes[0, 0].legend(fontsize=8)

steps = np.asarray(hist["eval_step"])
means = np.asarray(hist["eval_length"])
stds = np.asarray(hist["eval_std"])
axes[0, 1].plot(steps, means, "s-", color="seagreen", label="eval mean")
axes[0, 1].fill_between(steps, means - stds, means + stds, color="seagreen", alpha=0.2)
axes[0, 1].axhline(zero_eval["mean_length"], color="black", ls="-.", label="zéro-action")
axes[0, 1].axhline(initial_eval["mean_length"], color="orange", ls=":", label="initial")
axes[0, 1].axhline(random_eval["mean_length"], color="gray", ls="--", label="random")
axes[0, 1].set_title("Évaluation sur resets officiels")
axes[0, 1].set_xlabel("itération")
axes[0, 1].set_ylabel("longueur moyenne")
axes[0, 1].legend(fontsize=8)

axes[0, 2].plot(hist["imagined_before"], "o--", color="indianred", label="avant opt")
axes[0, 2].plot(hist["imagined_after"], "s-", color="darkred", label="meilleur accepté")
axes[0, 2].set_title(r"Coût imaginé $J(\pi)$")
axes[0, 2].set_xlabel("itération")
axes[0, 2].set_ylabel("coût moyen par pas")
axes[0, 2].legend(fontsize=8)

axes[1, 0].plot(hist["model_loss"], "o-", color="steelblue", label="train")
axes[1, 0].plot(hist["val_loss"], "s--", color="darkorange", label="validation")
axes[1, 0].set_title("Loss du modèle BNN")
axes[1, 0].set_xlabel("itération")
axes[1, 0].set_ylabel(r"MSE $\Delta s$")
axes[1, 0].legend(fontsize=8)

axes[1, 1].plot(hist["data_size"], "o-", color="black", label="transitions")
axes[1, 1].axhline(BUFFER_CAP, color="gray", ls=":", label="cap")
axes[1, 1].set_title("Dataset utilisé par le BNN")
axes[1, 1].set_xlabel("itération")
axes[1, 1].set_ylabel("transitions")
axes[1, 1].legend(fontsize=8)

axes[1, 2].plot(hist["mean_abs_angle"], "o-", color="purple", label="collecte")
if hist["eval_abs_angle"]:
    axes[1, 2].plot(steps, hist["eval_abs_angle"], "s--", color="mediumpurple", label="eval")
axes[1, 2].axhline(np.degrees(0.2), color="black", ls=":", label="seuil terminaison")
axes[1, 2].set_title("Distance angulaire à l'upright")
axes[1, 2].set_xlabel("itération")
axes[1, 2].set_ylabel("|θ| moyen (degrés)")
axes[1, 2].legend(fontsize=8)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharex=True)
axes[0].plot(final_trace["angles"], color="purple", label="angle final")
axes[0].axhline(0, color="green", ls="--", label="cible upright")
axes[0].axhline(np.degrees(0.2), color="black", ls=":", lw=1, label="terminaison")
axes[0].axhline(-np.degrees(0.2), color="black", ls=":", lw=1)
axes[0].set_title("Replay final : angle du pendule")
axes[0].set_xlabel("pas de temps")
axes[0].set_ylabel("angle θ (degrés)")
axes[0].legend(fontsize=8)

axes[1].plot(final_trace["actions"], color="teal", label="force")
axes[1].axhline(3, color="black", ls=":", lw=1)
axes[1].axhline(-3, color="black", ls=":", lw=1)
axes[1].set_title("Replay final : actions")
axes[1].set_xlabel("pas de temps")
axes[1].set_ylabel("force chariot")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lecture.** La métrique principale n'est plus le reward négatif de `Pendulum-v1`, mais la
**durée avant chute** : dans `InvertedPendulum-v5`, chaque pas vivant vaut `+1`, donc reward et
longueur portent la même information. Une amélioration réelle doit donc apparaître sur
l'**évaluation périodique à seeds fixes**, pas seulement sur la trajectoire collectée avec bruit
d'exploration.

La baseline zéro-action est importante : elle vérifie que le simulateur ne se stabilise pas tout
seul et que les petites forces finales correspondent bien à un contrôle actif. Le panneau du coût
imaginé sert, lui, à détecter si la politique trouve une direction *dans le modèle*. S'il baisse
alors que l'évaluation réelle plafonne, le BNN est probablement exploité hors distribution.

C'est ici qu'il faut lire le **gap imagined/real** :

- l'imagination optimise un horizon de **80 pas** sous un BNN approximatif ;
- le score réel demande de survivre jusqu'à **1000 pas** dans Gymnasium ;
- une baisse de $J(\pi)$ ne vaut preuve qu'après validation multi-seed réelle.

La recette retenue passe précisément ce test : la meilleure politique est choisie sur seeds fixes,
restaurée, puis confirmée sur **20 seeds held-out disjointes** à **$1000 \pm 0$**. Le point à
retenir n'est donc pas seulement « le coût imaginé baisse », mais « ce coût imaginé correspond à
une stabilisation réelle parfaitement reproductible sur le protocole final ».


In [ ]:
# Démonstration finale : replay vidéo notebook, sans fenêtre une fenêtre locale.

In [ ]:
def record_deep_policy_video(policy, *, label="deep_pilco_invertedpendulum", seed=0, max_steps=300, video_root="videos/10b_deep_pilco"):
    video_dir = Path(video_root) / label
    video_dir.mkdir(parents=True, exist_ok=True)
    env_video = gym.make(ENV_ID, render_mode="rgb_array")
    env_video = RecordVideo(
        env_video,
        video_folder=str(video_dir),
        name_prefix=label,
        episode_trigger=lambda episode_id: episode_id == 0,
        disable_logger=True,
    )
    try:
        total_return = rollout_deep_policy(env_video, policy, seed=seed, max_steps=max_steps)
    finally:
        env_video.close()
    return sorted(video_dir.glob("*.mp4")), total_return

print("✓ helpers InvertedPendulum prêts : encodage, projection, évaluation réelle, particules et vidéo")


In [ ]:
try:
    videos, total_return = record_deep_policy_video(
        policy,
        label="deep_pilco_invertedpendulum",
        seed=1,
        max_steps=EVAL_HORIZON,
    )
    print(f"Démo Deep PILCO | retour={total_return:.1f}")
    if videos:
        print("Replay vidéo Deep PILCO :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print("Aucune vidéo générée pour Deep PILCO.")
except Exception as exc:
    print("Démo vidéo non disponible :", type(exc).__name__, exc)


**Lecture de la démonstration visuelle.** La vidéo part du reset officiel de Gymnasium : le pendule
est déjà proche de l'upright, comme dans la vraie tâche `InvertedPendulum-v5`, et le modèle doit
éviter qu'il tombe. Elle ne remplace pas les courbes : un replay court peut être chanceux, alors
que l'évaluation multi-seed mesure la robustesse.

La hiérarchie de preuve est donc stricte : **vidéo < validation fixe < held-out final disjoint**.
Le replay illustre *comment* la politique stabilise ; le score held-out dit *si* cette
stabilisation tient vraiment. Et même avec un score final à `1000`, cela ne transforme pas la
tâche Gym upright en reproduction directe du swing-up du papier.


## Conclusion et ponts vers PETS / MBPO / Dreamer

### Ce qu'on a construit

Deep PILCO garde la boucle conceptuelle de PILCO, mais remplace ses briques analytiques par des briques
neuronales et échantillonnées :

1. **Observer** quelques trajectoires réelles ;
2. apprendre un modèle de dynamique neuronal $f_\phi(x,u)\approx \Delta x$ ;
3. représenter l'incertitude du modèle par **MC-dropout** ;
4. **imaginer** des trajectoires avec un nuage de particules ;
5. optimiser une politique différentiable par rétropropagation à travers ces particules ;
6. restaurer la meilleure politique selon une évaluation réelle tenue à part.

Le changement important n'est pas seulement “GP → réseau”. C'est tout le mode de calcul qui change :
PILCO propageait des moments gaussiens en forme fermée ; Deep PILCO propage des **échantillons**.

| Brique | Rôle |
|--------|------|
| BNN MC-dropout | Approxime une distribution sur les dynamiques possibles |
| Masque fixe par particule | Garde une hypothèse de dynamique cohérente sur tout l'horizon |
| Particules | Remplacent le moment matching analytique |
| Resampling par moments | Résume le nuage et stabilise les rollouts longs |
| Coût dense + barrière | Donne un signal de contrôle plus informatif près de la chute |
| Politique MLP | Plus expressive que la petite politique RBF |
| Best-restore | Protège contre les updates de politique bruitées ou régressives |

### Ce que Deep PILCO apporte

Deep PILCO répond à la limite la plus évidente de PILCO : un GP exact devient coûteux quand le dataset
grandit et se prête mal aux dynamiques plus hautes dimensions. Le réseau neuronal peut absorber plus de
données, apprendre des fonctions plus riches et s'entraîner par mini-batches.

Mais pour rester fidèle à l'esprit PILCO, le réseau ne doit pas être seulement déterministe. Il doit
porter une incertitude utilisable par la planification. Ici, le dropout joue ce rôle : chaque masque
définit une dynamique plausible, et les particules explorent plusieurs futurs possibles. Le point clé
est de garder le **même masque sur l'horizon** : sinon l'incertitude se moyenne artificiellement à
chaque pas et la trajectoire devient trop optimiste.

### Pourquoi nos résultats restent à lire prudemment

Le notebook montre que la recette peut vraiment apprendre sur InvertedPendulum, mais ce n'est pas une
reproduction complète du papier Deep PILCO. Notre version est calibrée pour rester lisible et exécutable
dans un notebook :

- le BNN est petit ;
- le nombre d'updates modèle/politique reste modéré ;
- l'horizon imaginé est plus court que l'horizon réel maximal ;
- le dropout donne une incertitude pratique, mais moins bien calibrée qu'un vrai posterior bayésien ;
- l'optimisation de politique est bruitée, car les particules et les masques changent le paysage de coût ;
- la sélection de la meilleure politique est indispensable : le dernier checkpoint n'est pas toujours le meilleur.

Le bon message n'est donc pas “Deep PILCO résout tout mieux que PILCO”. Le message est plus précis :
on peut conserver l'idée de PILCO — apprendre peu dans le vrai monde, améliorer beaucoup dans le modèle —
tout en remplaçant les calculs GP analytiques par un modèle neuronal probabiliste et des rollouts
échantillonnés.

### Limites structurelles de Deep PILCO

| Limite | Conséquence |
|--------|-------------|
| Incertitude approximée par dropout | Calibration fragile ; le choix de $p$ influence fortement le comportement |
| Rollouts en boucle ouverte | Les erreurs du modèle se composent avec l'horizon |
| Particules stochastiques | Le coût imaginé est bruité, donc l'optimisation de politique oscille davantage |
| Pas de replay off-policy pour la politique | Les données réelles servent au modèle, pas directement à un acteur off-policy |
| Coût défini à la main | Fonctionne bien sur InvertedPendulum, moins évident sur une tâche sans but simple |
| Best checkpoint nécessaire | Le dernier modèle peut être moins bon qu'une politique intermédiaire |

### Comparaison PILCO vs Deep PILCO

| Dimension | PILCO | Deep PILCO |
|----------|-------|------------|
| Modèle de dynamique | GP exact | BNN MC-dropout |
| Incertitude | Posterior GP analytique | Approximation par masques dropout |
| Propagation | Moment matching | Particules + resampling |
| Optimiseur politique | L-BFGS quasi-Newton | Adam / gradients stochastiques |
| Scalabilité données | Limitée par le GP | Meilleure grâce aux mini-batches |
| Stabilité | Souvent plus nette à petit dataset | Plus bruitée, mais plus flexible |
| Point pédagogique | Incertitude analytique | Incertitude échantillonnée et deep model-based RL |

### Pont vers la suite

Deep PILCO donne une première version “deep” du contrôle model-based, mais les méthodes suivantes
changeront encore la manière d'utiliser le modèle.

$$
\underbrace{\text{PILCO}}_{\substack{\text{GP exact} \\ \text{moments analytiques}}}
\quad\longrightarrow\quad
\underbrace{\text{Deep PILCO}}_{\substack{\text{BNN dropout} \\ \text{particules}}}
\quad\longrightarrow\quad
\underbrace{\text{PETS}}_{\substack{\text{ensemble probabiliste} \\ \text{MPC + CEM}}}
\quad\longrightarrow\quad
\underbrace{\text{MBPO}}_{\substack{\text{rollouts courts} \\ \text{SAC off-policy}}}
\quad\longrightarrow\quad
\underbrace{\text{Dreamer}}_{\substack{\text{monde latent} \\ \text{actor-critic imaginé}}}
$$

- **PETS** remplacera le dropout par un **ensemble** de réseaux probabilistes et planifiera à chaque pas
  avec MPC/CEM au lieu d'apprendre une politique fixe.
- **MBPO** utilisera le modèle pour générer de courts rollouts artificiels qui alimentent un agent
  off-policy comme SAC.
- **Dreamer** apprendra un monde latent différentiable et entraînera directement l'acteur et le critique
  dans l'imagination.

### Recette retenue dans ce notebook

Dans cette version, la recette qui fonctionne est volontairement documentée :

| Choix | Valeur / idée |
|------|---------------|
| Environnement | `InvertedPendulum-v5`, resets officiels Gymnasium |
| Politique | MLP continue |
| Modèle | BNN avec dropout |
| Dropout | $p=0.08$ |
| Particules | $96$ |
| Horizon imaginé | $80$ |
| Updates modèle / politique | $260 / 70$ |
| Objectif | coût dense + barrière près de la chute |
| Sélection | évaluation sur seeds fixes + restauration du meilleur |
| Mesure finale | seeds held-out disjointes |

> **Ce qu'il faut retenir.** Deep PILCO est le même pari que PILCO, mais avec des outils plus
> scalables : un world model neuronal, une incertitude approximée, et une propagation par particules.
> Cette flexibilité rend la méthode plus générale, mais aussi moins propre numériquement : l'incertitude
> est moins bien calibrée, les gradients sont plus bruités, et la qualité finale dépend beaucoup du
> protocole d'évaluation et de restauration du meilleur modèle.

## Références

## Références

- Gal, Y., McAllister, R. T. & Rasmussen, C. E. (2016). *Improving PILCO with Bayesian Neural Network Dynamics Models*. ICML Data-Efficient Machine Learning Workshop. [PDF](https://www.cs.ox.ac.uk/people/yarin.gal/website/PDFs/DeepPILCO.pdf).  
  **Référence centrale du notebook** : remplace le GP de PILCO par un BNN, utilise le dropout comme approximation bayésienne et propage l'incertitude par particules.

- McAllister, R. T. — page projet *Improving PILCO with Bayesian Neural Network Dynamics Models*. [Page](https://rowanmcallister.github.io/publication/dpilco/).  
  Résumé utile du papier : Deep PILCO vise à faire passer PILCO à l'échelle linéaire en nombre de trials et en dimension d'observation, en remplaçant le moment matching analytique par des méthodes particulaires.

- Gal, Y. & Ghahramani, Z. (2016). *Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning*. ICML. [Proceedings](https://proceedings.mlr.press/v48/gal16.html) / [arXiv:1506.02142](https://arxiv.org/abs/1506.02142).  
  Fondement théorique du **MC-dropout** : garder le dropout actif à l'inférence revient à approximer une distribution bayésienne sur les fonctions du réseau.

- Deisenroth, M. P. & Rasmussen, C. E. (2011). *PILCO: A Model-Based and Data-Efficient Approach to Policy Search*. ICML. [PDF](https://mlg.eng.cam.ac.uk/pub/pdf/DeiRas11.pdf).  
  Point de départ conceptuel : GP de dynamique, propagation d'incertitude, optimisation de politique avec très peu de données réelles.

- Rasmussen, C. E. & Williams, C. K. I. (2006). *Gaussian Processes for Machine Learning*. MIT Press. [Livre en ligne](http://www.gaussianprocess.org/gpml/).  
  Référence de fond pour comprendre ce que Deep PILCO remplace : posterior GP, kernels, prédiction probabiliste et coût du GP exact.

- Blundell, C., Cornebise, J., Kavukcuoglu, K. & Wierstra, D. (2015). *Weight Uncertainty in Neural Networks*. ICML. [arXiv:1505.05424](https://arxiv.org/abs/1505.05424).  
  Référence générale sur les **Bayesian Neural Networks** : distribution sur les poids, incertitude épistémique et apprentissage variationnel.

- Gymnasium / Farama Foundation. *InvertedPendulum-v5 documentation*. [Documentation](https://gymnasium.farama.org/environments/mujoco/inverted_pendulum/).  
  Source du protocole expérimental utilisé ici : observation, action continue, récompense `+1` par pas vivant et terminaison à `|angle| > 0.2`.

- Todorov, E., Erez, T. & Tassa, Y. (2012). *MuJoCo: A physics engine for model-based control*. IROS. [IEEE](https://ieeexplore.ieee.org/document/6386109).  
  Référence du simulateur physique derrière `InvertedPendulum-v5` et les tâches MuJoCo utilisées dans la suite.